# Experiment 1: Model Scale Analysis (RQ1)
## Does model scale correlate with survival duration in embodied environments?

**Models:** Qwen2.5 family at 4 scales (1.5B, 3B, 7B, 14B)
**Environments:** Forest, Desert, Tundra, Hunt
**Seeds:** 0-9 per configuration
**Reasoning Strategy:** Structured planning (5-step protocol)

**Hypothesis:** Larger models survive longer due to improved multi-objective reasoning, with diminishing returns beyond 7B.

In [ ]:
# Install dependencies and clone repository
!git clone https://github.com/ritwikraha/earth_33.git
%cd earth_33
!pip install -e ".[recording]"
!pip install transformers accelerate bitsandbytes huggingface_hub torch sentencepiece protobuf

In [ ]:
from huggingface_hub import login

hf_read_key = 'some_api_key'
login(token=hf_read_key)

In [ ]:
import os
import json
import time
import logging
import warnings
warnings.filterwarnings('ignore')

# Headless mode for Colab (no display)
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# Experiment 1: Model Scale Analysis
# Test Qwen2.5 family across 4 scales

MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
]

CONFIGS = ["configs/forest.yaml", "configs/desert.yaml", "configs/tundra.yaml", "configs/hunt.yaml"]
SEEDS = list(range(10))

# Results storage
results = {}

In [ ]:
import traceback
from agents.hf_agent import HuggingFaceAgent
from config_io.config import load_config
from eval.runner import run_episode
from eval.metrics import compute_metrics
def run_single_experiment(model_name, config_path, seed, agent=None):
    """Run a single episode and return metrics."""
    config = load_config(config_path)
    if agent is None:
        agent = HuggingFaceAgent(
            model_name=model_name,
            quantize_4bit=True,
            temperature=0.7,
            max_new_tokens=256,
            reasoning_strategy="structured",
        )
    agent.reset()

    replay_data = run_episode(config, seed, agent, headless=True)
    metrics = compute_metrics(replay_data)
    metrics["seed"] = seed
    metrics["model"] = model_name
    metrics["config"] = os.path.basename(config_path).replace(".yaml", "")
    metrics["agent_stats"] = agent.stats
    return metrics
# Run all experiments
for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Loading model: {model_name}")
    print(f"{'='*60}")

    # Load agent once per model (reuse across configs/seeds)
    agent = HuggingFaceAgent(
        model_name=model_name,
        quantize_4bit=True,
        temperature=0.7,
        max_new_tokens=256,
        reasoning_strategy="structured",
    )

    model_key = model_name.split("/")[-1]
    results[model_key] = []

    for config_path in CONFIGS:
        config_name = os.path.basename(config_path).replace(".yaml", "")
        print(f"\n  Config: {config_name}")

        for seed in SEEDS:
            try:
                metrics = run_single_experiment(model_name, config_path, seed, agent=agent)
                results[model_key].append(metrics)
                print(f"    Seed {seed}: {metrics['survived_steps']} steps, "
                      f"death: {metrics['cause_of_death']}")
            except Exception as e:
                err_msg = repr(e)  # repr() never returns empty
                tb = traceback.format_exc()
                print(f"    Seed {seed}: FAILED - {err_msg}")
                print(f"    Traceback: {tb[-500:]}")  # last 500 chars of traceback
                results[model_key].append({
                    "seed": seed, "model": model_name,
                    "config": config_name, "survived_steps": 0,
                    "cause_of_death": f"ERROR: {err_msg[:200]}",
                    "error": True,
                })

    # Free GPU memory before loading next model
    del agent
    import torch
    torch.cuda.empty_cache()
    import gc
    gc.collect()
print("\n\nAll experiments complete!")


In [ ]:
# Save raw results
output_dir = "runs/experiment_1_model_scale"
os.makedirs(output_dir, exist_ok=True)

with open(f"{output_dir}/raw_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {output_dir}/raw_results.json")

# Print summary table
print("\n" + "="*80)
print("EXPERIMENT 1 SUMMARY: Model Scale Analysis")
print("="*80)

for model_key, model_results in results.items():
    valid_results = [r for r in model_results if not r.get("error")]
    if not valid_results:
        print(f"\n{model_key}: No valid results")
        continue

    print(f"\n{model_key}:")
    for config_name in ["forest", "desert", "tundra", "hunt"]:
        config_results = [r for r in valid_results if r["config"] == config_name]
        if config_results:
            avg_steps = sum(r["survived_steps"] for r in config_results) / len(config_results)
            deaths = {}
            for r in config_results:
                cod = r["cause_of_death"]
                deaths[cod] = deaths.get(cod, 0) + 1
            top_death = max(deaths, key=deaths.get) if deaths else "N/A"
            print(f"  {config_name:8s}: avg={avg_steps:6.1f} steps | "
                  f"top death: {top_death} ({deaths.get(top_death, 0)}/{len(config_results)})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
config_names = ["forest", "desert", "tundra", "hunt"]
model_names = list(results.keys())
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))

for ax_idx, config_name in enumerate(config_names):
    ax = axes[ax_idx]
    means = []
    stds = []
    labels = []

    for model_key in model_names:
        valid = [r for r in results.get(model_key, [])
                 if r.get("config") == config_name and not r.get("error")]
        if valid:
            steps = [r["survived_steps"] for r in valid]
            means.append(np.mean(steps))
            stds.append(np.std(steps))
        else:
            means.append(0)
            stds.append(0)
        labels.append(model_key.replace("Qwen2.5-", "").replace("-Instruct", ""))

    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_title(config_name.capitalize(), fontsize=12, fontweight='bold')
    if ax_idx == 0:
        ax.set_ylabel("Avg. Survival Steps", fontsize=11)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle("Experiment 1: Model Scale vs. Survival Duration", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{output_dir}/scale_vs_survival.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{output_dir}/scale_vs_survival.pdf", dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {output_dir}/scale_vs_survival.png")

In [ ]:
fig, axes = plt.subplots(len(model_names), 1, figsize=(12, 3 * len(model_names)))

death_types = ["DEHYDRATION", "STARVATION", "HYPOTHERMIA", "HYPERTHERMIA", "TRAUMA", "INFECTION", "HUNTED"]
death_colors = {'DEHYDRATION': '#2196F3', 'STARVATION': '#FF9800', 'HYPOTHERMIA': '#00BCD4',
                'HYPERTHERMIA': '#F44336', 'TRAUMA': '#9C27B0', 'INFECTION': '#4CAF50', 'HUNTED': '#795548'}

for m_idx, model_key in enumerate(model_names):
    ax = axes[m_idx] if len(model_names) > 1 else axes
    valid = [r for r in results.get(model_key, []) if not r.get("error")]

    bottom = np.zeros(4)
    for dt in death_types:
        counts = []
        for config_name in config_names:
            config_results = [r for r in valid if r["config"] == config_name]
            count = sum(1 for r in config_results if r.get("cause_of_death") == dt)
            counts.append(count)
        ax.bar(config_names, counts, bottom=bottom, label=dt, color=death_colors.get(dt, 'gray'))
        bottom += np.array(counts)

    ax.set_title(model_key, fontsize=11)
    ax.set_ylabel("Count")
    if m_idx == 0:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

fig.suptitle("Cause of Death Distribution by Model Scale", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{output_dir}/death_causes.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{output_dir}/death_causes.pdf", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scaling curve: parameter count vs. average survival
param_counts = {"Qwen2.5-1.5B-Instruct": 1.5, "Qwen2.5-3B-Instruct": 3.0,
                "Qwen2.5-7B-Instruct": 7.0, "Qwen2.5-14B-Instruct": 14.0}

fig, ax = plt.subplots(figsize=(8, 5))

for config_name in config_names:
    params = []
    avg_steps = []
    for model_key in model_names:
        valid = [r for r in results.get(model_key, [])
                 if r.get("config") == config_name and not r.get("error")]
        if valid and model_key in param_counts:
            params.append(param_counts[model_key])
            avg_steps.append(np.mean([r["survived_steps"] for r in valid]))

    ax.plot(params, avg_steps, 'o-', label=config_name.capitalize(), markersize=8, linewidth=2)

ax.set_xlabel("Parameters (Billions)", fontsize=12)
ax.set_ylabel("Avg. Survival Steps", fontsize=12)
ax.set_title("Scaling Curve: Model Size vs. Survival Duration", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xscale('log')
ax.set_xticks([1.5, 3, 7, 14])
ax.set_xticklabels(['1.5B', '3B', '7B', '14B'])

plt.tight_layout()
plt.savefig(f"{output_dir}/scaling_curve.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{output_dir}/scaling_curve.pdf", dpi=150, bbox_inches='tight')
plt.show()

## Generate GIFs for Paper Figures

Run individual episodes with recording to generate visual demonstrations for the paper.

In [ ]:
# Generate GIF recordings for best/worst performing seeds
# Note: This requires pygame display. On Colab, we run headless and save replays.
# GIFs can be generated locally using:
#   python -m cli run_episode --config configs/hunt.yaml --agent hf --seed 42 --record runs/demo.gif

# For Colab, save replay JSONs that can be converted to GIFs locally
for model_name in [MODELS[0], MODELS[-1]]:  # Smallest and largest
    model_key = model_name.split("/")[-1]
    agent = HuggingFaceAgent(
        model_name=model_name,
        quantize_4bit=True,
        temperature=0.7,
        reasoning_strategy="structured",
    )

    config = Config.from_yaml("configs/hunt.yaml")
    replay_path = f"{output_dir}/replay_{model_key}_seed0.json"
    replay_data = run_episode(config, 0, agent, headless=True, replay_path=replay_path)
    print(f"Replay saved: {replay_path}")

    del agent
    import torch
    torch.cuda.empty_cache()
    import gc
    gc.collect()

In [ ]:
# Zip results for download
import shutil
shutil.make_archive(f"/content/experiment_1_results", 'zip', output_dir)
print("Download: /content/experiment_1_results.zip")

# If running on Colab, trigger download
try:
    from google.colab import files
    files.download('/content/experiment_1_results.zip')
except ImportError:
    print("Not running on Colab. Results at:", output_dir)